In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import ConfusionMatrixDisplay

ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, ROOT)
from src.split_data import split_data

In [ ]:
features_path = os.path.join(ROOT, "data", "features.csv")
splits_dir    = os.path.join(ROOT, "data", "splits")
train_df, val_df, test_df = split_data(csv_path=features_path, train_pct=0.65, val_pct=0.20, seed=42, output_dir=splits_dir)

In [ ]:
cancerous = {"BCC", "SCC", "MEL"}
base_drops = ["img_id", "diagnostic", "Unnamed: 0"]

scenarios = {
    "none":                  ["contrast_diff", "contrast_ratio", "contrast_standardized"],
    "contrast_diff":         ["contrast_ratio", "contrast_standardized"],
    "contrast_ratio":        ["contrast_diff", "contrast_standardized"],
    "contrast_standardized": ["contrast_diff", "contrast_ratio"],
    "all":                   [],
}

train_labels = np.where(train_df["diagnostic"].isin(cancerous), "cancerous", "non-cancerous")
val_labels   = np.where(val_df["diagnostic"].isin(cancerous),   "cancerous", "non-cancerous")
test_labels  = np.where(test_df["diagnostic"].isin(cancerous),  "cancerous", "non-cancerous")

results = {}
for name, extra_drops in scenarios.items():
    drops = base_drops + extra_drops
    train_features = train_df.drop(columns=drops, errors="ignore").select_dtypes(include="number")
    val_features   = val_df[train_features.columns]
    test_features  = test_df[train_features.columns]

    forest = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=6)
    forest.fit(train_features, train_labels)

    results[name] = {
        "val_preds":  forest.predict(val_features),
        "test_preds": forest.predict(test_features),
        "val_acc":    forest.score(val_features, val_labels),
        "test_acc":   forest.score(test_features, test_labels),
    }
    print(f"{name:25s}  val={results[name]['val_acc']:.3f}  test={results[name]['test_acc']:.3f}")

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(9, 22))
for i, (name, r) in enumerate(results.items()):
    ConfusionMatrixDisplay.from_predictions(val_labels,  r["val_preds"],  cmap="Blues", ax=axes[i, 0], colorbar=False)
    ConfusionMatrixDisplay.from_predictions(test_labels, r["test_preds"], cmap="Blues", ax=axes[i, 1], colorbar=False)
    axes[i, 0].set_title(f"{name} — val (acc={r['val_acc']:.3f})")
    axes[i, 1].set_title(f"{name} — test (acc={r['test_acc']:.3f})")
plt.tight_layout()
plt.show()